# Exercice 05 - Stockage dans MinIO (Data Lake)

## Objectifs
- Configurer Spark pour se connecter a MinIO
- Lire et ecrire des donnees dans MinIO
- Organiser les donnees en Bronze / Silver / Gold
- Comprendre les chemins S3

---

## 1. Rappel : Architecture du Data Lake

```
+------------------------------------------------------------------+
|                         MINIO (S3)                               |
+------------------------------------------------------------------+
|                                                                  |
|  +----------------+  +----------------+  +----------------+      |
|  |    BRONZE      |  |    SILVER      |  |     GOLD       |      |
|  |                |  |                |  |                |      |
|  | Donnees brutes |  | Donnees        |  | Donnees        |      |
|  | telles que     |  | nettoyees      |  | agregees       |      |
|  | recues         |  | et typees      |  | pret pour      |      |
|  |                |  |                |  | l'analyse      |      |
|  | s3a://bronze/  |  | s3a://silver/  |  | s3a://gold/    |      |
|  +----------------+  +----------------+  +----------------+      |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration de Spark pour MinIO

In [ ]:
from pyspark.sql import SparkSession

# Arreter la session existante si elle existe (necessaire pour charger les JARs)
try:
    spark.stop()
except:
    pass

# Configuration de Spark pour se connecter a MinIO
# Les JARs AWS sont telecharges automatiquement au premier lancement
spark = SparkSession.builder \
    .appName("Spark MinIO") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("Spark configure pour MinIO !")
print(f"Version: {spark.version}")

### Explication de la configuration

| Option | Description |
|--------|-------------|
| `fs.s3a.endpoint` | Adresse du serveur MinIO |
| `fs.s3a.access.key` | Identifiant de connexion |
| `fs.s3a.secret.key` | Mot de passe |
| `fs.s3a.path.style.access` | Necessaire pour MinIO |
| `fs.s3a.impl` | Implementation du systeme de fichiers S3 |

## 3. Creation des buckets

In [ ]:
pip install minio

In [ ]:
from minio import Minio

# Connexion a MinIO
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

# Creer les buckets s'ils n'existent pas
for bucket in ["bronze", "silver", "gold"]:
    if not client.bucket_exists(bucket):
        client.make_bucket(bucket)
        print(f"Bucket '{bucket}' cree")
    else:
        print(f"Bucket '{bucket}' existe deja")

## 4. Creer des donnees de test

In [ ]:
# Creer un DataFrame de ventes
ventes = [
    ("2025-01-01", "Laptop", "Paris", 2, 999.99),
    ("2025-01-01", "Souris", "Lyon", 10, 29.99),
    ("2025-01-02", "Clavier", "Paris", 5, 79.99),
    ("2025-01-02", "Ecran", "Marseille", 3, 299.99),
    ("2025-01-03", "Laptop", "Lyon", 1, 999.99),
    ("2025-01-03", "Casque", "Paris", 8, 149.99)
]

colonnes = ["date", "produit", "ville", "quantite", "prix_unitaire"]

df_ventes = spark.createDataFrame(ventes, colonnes)
df_ventes.show()

## 5. Ecriture dans Bronze

Les chemins S3 ont le format : `s3a://bucket/chemin/fichier`

In [ ]:
# Ecrire les donnees brutes dans Bronze
chemin_bronze = "s3a://bronze/ventes/raw"

df_ventes.write \
    .mode("overwrite") \
    .parquet(chemin_bronze)

print(f"Donnees ecrites dans : {chemin_bronze}")

In [ ]:
# Verifier en relisant
df_bronze = spark.read.parquet(chemin_bronze)
print("Lecture depuis Bronze :")
df_bronze.show()

## 6. Transformation vers Silver

En Silver, on nettoie et on enrichit les donnees.

In [ ]:
from pyspark.sql.functions import col, to_date

# Transformation : ajouter le montant total
df_silver = df_bronze \
    .withColumn("date", to_date(col("date"))) \
    .withColumn("montant_total", col("quantite") * col("prix_unitaire"))

df_silver.show()
df_silver.printSchema()

In [ ]:
# Ecrire dans Silver
chemin_silver = "s3a://silver/ventes/enriched"

df_silver.write \
    .mode("overwrite") \
    .parquet(chemin_silver)

print(f"Donnees ecrites dans : {chemin_silver}")

## 7. Agregation vers Gold

En Gold, on prepare les donnees pour l'analyse.

In [ ]:
from pyspark.sql.functions import sum, count, avg

# Lire depuis Silver
df_silver = spark.read.parquet(chemin_silver)

# Agregation par ville
df_gold_ville = df_silver.groupBy("ville").agg(
    count("*").alias("nb_ventes"),
    sum("quantite").alias("total_quantite"),
    sum("montant_total").alias("chiffre_affaires"),
    avg("montant_total").alias("panier_moyen")
)

df_gold_ville.show()

In [ ]:
# Ecrire dans Gold
chemin_gold = "s3a://gold/ventes/par_ville"

df_gold_ville.write \
    .mode("overwrite") \
    .parquet(chemin_gold)

print(f"Donnees ecrites dans : {chemin_gold}")

## 8. Verification du Data Lake

Listons les objets dans chaque bucket.

In [ ]:
# Lister les objets dans chaque bucket
for bucket in ["bronze", "silver", "gold"]:
    print(f"\n=== Bucket: {bucket} ===")
    objets = client.list_objects(bucket, recursive=True)
    for obj in objets:
        print(f"  {obj.object_name} ({obj.size} octets)")

## 9. Schema du pipeline

```
DONNEES SOURCES                      DATA LAKE (MinIO)
---------------                      -----------------

  [DataFrame]                        +-------------------+
       |                             |      BRONZE       |
       |  Ingestion brute            |  s3a://bronze/    |
       +--------------------------->|  ventes/raw       |
                                     +--------+----------+
                                              |
                                              | Nettoyage
                                              | Enrichissement
                                              v
                                     +-------------------+
                                     |      SILVER       |
                                     |  s3a://silver/    |
                                     |  ventes/enriched  |
                                     +--------+----------+
                                              |
                                              | Agregation
                                              | Indicateurs
                                              v
                                     +-------------------+
                                     |       GOLD        |
                                     |  s3a://gold/      |
                                     |  ventes/par_ville |
                                     +-------------------+
                                              |
                                              v
                                        Reporting
                                        BI Tools
```

---

## Exercice

**Objectif** : Creer votre propre pipeline Bronze / Silver / Gold

**Consigne** :
1. Creez un DataFrame de clients (id, nom, email, pays, date_inscription)
2. Ecrivez-le dans `s3a://bronze/clients/raw`
3. Transformez : ajoutez une colonne `annee_inscription` extraite de la date
4. Ecrivez dans `s3a://silver/clients/enriched`
5. Agregez : comptez les clients par pays
6. Ecrivez dans `s3a://gold/clients/par_pays`

A vous de jouer :

In [ ]:
# Donnees clients
clients = [
    (1, "Alice Martin", "alice@email.com", "France", "2023-03-15"),
    (2, "Bob Johnson", "bob@email.com", "USA", "2023-06-22"),
    (3, "Charlie Dupont", "charlie@email.com", "France", "2024-01-10"),
    (4, "Diana Smith", "diana@email.com", "UK", "2023-11-05"),
    (5, "Eve Bernard", "eve@email.com", "France", "2024-02-28")
]

colonnes = ["id", "nom", "email", "pays", "date_inscription"]

# TODO: Creez le DataFrame

# TODO: Ecrivez dans Bronze

In [ ]:
# TODO: Transformez et ecrivez dans Silver
# Indice : utilisez year(to_date(col("date_inscription"))) pour extraire l'annee

from pyspark.sql.functions import year, to_date, col

In [ ]:
# TODO: Agregez et ecrivez dans Gold

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **configurer Spark pour MinIO**
- Le format des chemins S3 : `s3a://bucket/chemin`
- Comment **lire et ecrire** dans MinIO avec Spark
- L'organisation **Bronze / Silver / Gold**

### Prochaine etape
Dans le prochain notebook, nous apprendrons a ingerer des donnees depuis **PostgreSQL**.